# Shared Train/Validation/Test Split for Predictive Models

In [1]:
from datetime import datetime

import polars as pl

from run_config import (
    PATHS,
    RUN_MODE,
    MODEL_START_DATE,
    MODEL_END_DATE,
)

## Train Test Split

The input and output paths are selected by `RUN_MODE` in `run_config.py`. Thus, both `sample` and `full` mode write to separate locations. The generated 70/15/15 train, validation and test files are shared inputs for all predictive models. The random split is grouped by calendar date, so all spatial units and time buckets of a day remain in the same split. The checks below verify disjoint dates and report complete-day counts plus zero- and positive-demand coverage for every partition.

In [2]:
DATASETS = (
    PATHS.gold_1h_demand_hexagon,
    PATHS.gold_1h_demand_census_tracts,
    PATHS.gold_1h_demand_community_areas,
    PATHS.gold_1h_demand_community_area_unfiltered,
    PATHS.gold_2h_demand_hexagon,
    PATHS.gold_2h_demand_census_tracts,
    PATHS.gold_2h_demand_community_areas,
    PATHS.gold_2h_demand_community_area_unfiltered,
    PATHS.gold_4h_demand_hexagon,
    PATHS.gold_4h_demand_census_tracts,
    PATHS.gold_4h_demand_community_areas,
    PATHS.gold_4h_demand_community_area_unfiltered,
)
OUTPUT_DIR = PATHS.train_test_dir
TARGET_COL = "trip_count"

SEED = 42
RANDOM = True

MODEL_START_TS = datetime.fromisoformat(MODEL_START_DATE)
MODEL_END_TS = datetime.fromisoformat(MODEL_END_DATE)
if MODEL_START_TS >= MODEL_END_TS:
    raise ValueError(
        f"MODEL_START_DATE must be before MODEL_END_DATE: "
        f"{MODEL_START_DATE} >= {MODEL_END_DATE}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run mode: {RUN_MODE}")
if RUN_MODE == "full":
    print(f"Model period: [{MODEL_START_DATE}, {MODEL_END_DATE})")
else:
    print("Model-period filter disabled in sample mode")
print(f"Inputs: {[path.name for path in DATASETS]}")
print(f"Output directory: {OUTPUT_DIR}")

Run mode: full
Model period: [2025-01-01T00:00:00, 2026-05-01T00:00:00)
Inputs: ['GOLD_1H_DEMAND_HEXAGON_7.parquet', 'GOLD_1H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_1H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet', 'GOLD_2H_DEMAND_HEXAGON_7.parquet', 'GOLD_2H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_2H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_2H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet', 'GOLD_4H_DEMAND_HEXAGON_7.parquet', 'GOLD_4H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_4H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_4H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet']
Output directory: /Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data


In [3]:
def create_splits(dataset_path):
    df_split = pl.scan_parquet(dataset_path)

    if RUN_MODE == "full":
        df_split = df_split.filter(
            (pl.col("datetime_hour") >= MODEL_START_TS)
            & (pl.col("datetime_hour") < MODEL_END_TS)
        )

    if RANDOM:
        # Keep every spatial unit and time bucket from the same calendar day
        # in one split. Unique dates are ordered reproducibly by their hash.
        # Validation and test receive exactly the same number of complete days.
        date_assignment = (
            df_split
            .select(pl.col("datetime_hour").dt.date().alias("_split_date"))
            .unique()
            .with_columns(
                pl.col("_split_date").hash(seed=SEED).alias("_split_order")
            )
            .sort(["_split_order", "_split_date"])
            .collect()
            .with_row_index("_date_rank")
        )
        n_dates = date_assignment.height
        n_holdout_dates = round(n_dates * 0.15)
        n_train_dates = n_dates - 2 * n_holdout_dates
        if n_train_dates <= 0 or n_holdout_dates <= 0:
            raise ValueError(f"Not enough dates for grouped 70/15/15 split: {n_dates}")

        date_assignment = (
            date_assignment
            .with_columns(
                pl.when(pl.col("_date_rank") < n_train_dates)
                .then(pl.lit("train"))
                .when(pl.col("_date_rank") < n_train_dates + n_holdout_dates)
                .then(pl.lit("val"))
                .otherwise(pl.lit("test"))
                .alias("_split")
            )
            .select(["_split_date", "_split"])
        )
        bucketed = (
            df_split
            .with_columns(
                pl.col("datetime_hour").dt.date().alias("_split_date")
            )
            .join(date_assignment.lazy(), on="_split_date", how="inner")
        )
        train = bucketed.filter(pl.col("_split") == "train")
        val = bucketed.filter(pl.col("_split") == "val")
        test = bucketed.filter(pl.col("_split") == "test")
        helper_columns = ["_split_date", "_split"]
        train = train.drop(helper_columns)
        val = val.drop(helper_columns)
        test = test.drop(helper_columns)
    else:
        train = df_split.filter(pl.col("datetime_hour") < pl.datetime(2025, 9, 1))
        val = df_split.filter(
            (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1))
            & (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
        )
        test = df_split.filter(pl.col("datetime_hour") >= pl.datetime(2026, 1, 1))

    return df_split, train, val, test


split_results = {}
for dataset_path in DATASETS:
    df_split, train, val, test = create_splits(dataset_path)
    output_paths = {
        "train": OUTPUT_DIR / f"{dataset_path.stem}_TRAIN.parquet",
        "val": OUTPUT_DIR / f"{dataset_path.stem}_VAL.parquet",
        "test": OUTPUT_DIR / f"{dataset_path.stem}_TEST.parquet",
    }

    counts = {
        "total": df_split.select(pl.len()).collect().item(),
        "train": train.select(pl.len()).collect().item(),
        "val": val.select(pl.len()).collect().item(),
        "test": test.select(pl.len()).collect().item(),
    }
    if counts["total"] == 0:
        raise ValueError(f"Input dataset is empty: {dataset_path}")
    if counts["train"] + counts["val"] + counts["test"] != counts["total"]:
        raise ValueError(f"Split counts do not add up for {dataset_path}")

    split_dates = {
        name: frame.select(
            pl.col("datetime_hour").dt.date().alias("date")
        ).unique().collect()
        for name, frame in {"train": train, "val": val, "test": test}.items()
    }
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = split_dates[left].join(split_dates[right], on="date", how="inner")
        if overlap.height:
            raise ValueError(f"Date leakage between {left} and {right}: {overlap.height} dates")
    date_counts = {name: dates.height for name, dates in split_dates.items()}

    target_stats = {}
    for name, frame in {"train": train, "val": val, "test": test}.items():
        stats = frame.select(
            pl.col(TARGET_COL).is_null().sum().alias("null_targets"),
            (pl.col(TARGET_COL) == 0).sum().alias("zero_demand"),
            (pl.col(TARGET_COL) > 0).sum().alias("positive_demand"),
            pl.col(TARGET_COL).min().alias("min_demand"),
            pl.col(TARGET_COL).mean().alias("mean_demand"),
            pl.col(TARGET_COL).max().alias("max_demand"),
        ).collect().row(0, named=True)
        if stats["null_targets"]:
            raise ValueError(
                f"{dataset_path.name} {name} contains null target values"
            )
        if stats["min_demand"] < 0:
            raise ValueError(
                f"{dataset_path.name} {name} contains negative demand"
            )
        if stats["zero_demand"] == 0 or stats["positive_demand"] == 0:
            raise ValueError(
                f"{dataset_path.name} {name} must contain both zero- and "
                "positive-demand observations"
            )
        target_stats[name] = stats

    train.sink_parquet(output_paths["train"])
    val.sink_parquet(output_paths["val"])
    test.sink_parquet(output_paths["test"])
    split_results[dataset_path.stem] = {
        "counts": counts,
        "date_counts": date_counts,
        "target_stats": target_stats,
        "paths": output_paths,
    }

    shares = {name: round(count / counts["total"], 2) for name, count in counts.items() if name != "total"}
    print(f"{dataset_path.name}: {counts}, shares={shares}")
    print(f"Grouped split dates: {date_counts}")
    print(f"Target coverage: {target_stats}")
    print(f"Written: {output_paths}")

GOLD_1H_DEMAND_HEXAGON_7.parquet: {'total': 1897320, 'train': 1326168, 'val': 285576, 'test': 285576}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 1261315, 'positive_demand': 64853, 'min_demand': 0, 'mean_demand': 1.8778744472796811, 'max_demand': 490}, 'val': {'null_targets': 0, 'zero_demand': 271111, 'positive_demand': 14465, 'min_demand': 0, 'mean_demand': 1.9716257668711656, 'max_demand': 434}, 'test': {'null_targets': 0, 'zero_demand': 271481, 'positive_demand': 14095, 'min_demand': 0, 'mean_demand': 1.9364862593495251, 'max_demand': 431}}
Written: {'train': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_TRAIN.parquet'), 'val': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/tra

GOLD_1H_DEMAND_CENSUS_TRACTS.parquet: {'total': 10219920, 'train': 7143408, 'val': 1538256, 'test': 1538256}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 6995093, 'positive_demand': 148315, 'min_demand': 0, 'mean_demand': 0.348625893971057, 'max_demand': 325}, 'val': {'null_targets': 0, 'zero_demand': 1504607, 'positive_demand': 33649, 'min_demand': 0, 'mean_demand': 0.36603075170842825, 'max_demand': 254}, 'test': {'null_targets': 0, 'zero_demand': 1506054, 'positive_demand': 32202, 'min_demand': 0, 'mean_demand': 0.3595071301525884, 'max_demand': 250}}
Written: {'train': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_CENSUS_TRACTS_TRAIN.parquet'), 'val': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA

GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet: {'total': 896280, 'train': 626472, 'val': 134904, 'test': 134904}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 572986, 'positive_demand': 53486, 'min_demand': 0, 'mean_demand': 3.9752407130725715, 'max_demand': 399}, 'val': {'null_targets': 0, 'zero_demand': 123124, 'positive_demand': 11780, 'min_demand': 0, 'mean_demand': 4.173701298701299, 'max_demand': 374}, 'test': {'null_targets': 0, 'zero_demand': 123357, 'positive_demand': 11547, 'min_demand': 0, 'mean_demand': 4.0993150684931505, 'max_demand': 373}}
Written: {'train': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_COMMUNITY_AREAS_TRAIN.parquet'), 'val': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/

GOLD_1H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet: {'total': 896280, 'train': 626472, 'val': 134904, 'test': 134904}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 277101, 'positive_demand': 349371, 'min_demand': 0, 'mean_demand': 8.722715460547319, 'max_demand': 524}, 'val': {'null_targets': 0, 'zero_demand': 59883, 'positive_demand': 75021, 'min_demand': 0, 'mean_demand': 9.021645021645021, 'max_demand': 496}, 'test': {'null_targets': 0, 'zero_demand': 59679, 'positive_demand': 75225, 'min_demand': 0, 'mean_demand': 8.8343043942359, 'max_demand': 505}}
Written: {'train': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_COMMUNITY_AREAS_UNFILTERED_TRAIN.parquet'), 'val': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/

GOLD_2H_DEMAND_HEXAGON_7.parquet: {'total': 948660, 'train': 663084, 'val': 142788, 'test': 142788}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 623778, 'positive_demand': 39306, 'min_demand': 0, 'mean_demand': 3.7557488945593622, 'max_demand': 848}, 'val': {'null_targets': 0, 'zero_demand': 134090, 'positive_demand': 8698, 'min_demand': 0, 'mean_demand': 3.9432515337423313, 'max_demand': 860}, 'test': {'null_targets': 0, 'zero_demand': 134270, 'positive_demand': 8518, 'min_demand': 0, 'mean_demand': 3.8729725186990502, 'max_demand': 843}}
Written: {'train': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_2H_DEMAND_HEXAGON_7_TRAIN.parquet'), 'val': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_te

GOLD_2H_DEMAND_CENSUS_TRACTS.parquet: {'total': 5109960, 'train': 3571704, 'val': 769128, 'test': 769128}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 3478975, 'positive_demand': 92729, 'min_demand': 0, 'mean_demand': 0.697251787942114, 'max_demand': 602}, 'val': {'null_targets': 0, 'zero_demand': 748197, 'positive_demand': 20931, 'min_demand': 0, 'mean_demand': 0.7320615034168565, 'max_demand': 488}, 'test': {'null_targets': 0, 'zero_demand': 749067, 'positive_demand': 20061, 'min_demand': 0, 'mean_demand': 0.7190142603051768, 'max_demand': 478}}
Written: {'train': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_2H_DEMAND_CENSUS_TRACTS_TRAIN.parquet'), 'val': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/f

GOLD_2H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet: {'total': 448140, 'train': 313236, 'val': 67452, 'test': 67452}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 97461, 'positive_demand': 215775, 'min_demand': 0, 'mean_demand': 17.445430921094637, 'max_demand': 982}, 'val': {'null_targets': 0, 'zero_demand': 20995, 'positive_demand': 46457, 'min_demand': 0, 'mean_demand': 18.043290043290042, 'max_demand': 990}, 'test': {'null_targets': 0, 'zero_demand': 21001, 'positive_demand': 46451, 'min_demand': 0, 'mean_demand': 17.6686087884718, 'max_demand': 999}}
Written: {'train': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_2H_DEMAND_COMMUNITY_AREAS_UNFILTERED_TRAIN.parquet'), 'val': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/

GOLD_4H_DEMAND_CENSUS_TRACTS.parquet: {'total': 2554980, 'train': 1785852, 'val': 384564, 'test': 384564}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 1725908, 'positive_demand': 59944, 'min_demand': 0, 'mean_demand': 1.394503575884228, 'max_demand': 1056}, 'val': {'null_targets': 0, 'zero_demand': 371093, 'positive_demand': 13471, 'min_demand': 0, 'mean_demand': 1.464123006833713, 'max_demand': 841}, 'test': {'null_targets': 0, 'zero_demand': 371609, 'positive_demand': 12955, 'min_demand': 0, 'mean_demand': 1.4380285206103536, 'max_demand': 897}}
Written: {'train': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_4H_DEMAND_CENSUS_TRACTS_TRAIN.parquet'), 'val': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/f

GOLD_4H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet: {'total': 224070, 'train': 156618, 'val': 33726, 'test': 33726}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 30271, 'positive_demand': 126347, 'min_demand': 0, 'mean_demand': 34.890861842189274, 'max_demand': 1897}, 'val': {'null_targets': 0, 'zero_demand': 6541, 'positive_demand': 27185, 'min_demand': 0, 'mean_demand': 36.086580086580085, 'max_demand': 1795}, 'test': {'null_targets': 0, 'zero_demand': 6518, 'positive_demand': 27208, 'min_demand': 0, 'mean_demand': 35.3372175769436, 'max_demand': 1803}}
Written: {'train': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_4H_DEMAND_COMMUNITY_AREAS_UNFILTERED_TRAIN.parquet'), 'val': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment

In [4]:
df_split.head(10).collect()

datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,date,is_holiday,community_area,weather_station_distance_km,food_drink,landmark,shop,train_station,tmpc,relh,sknt,p01m,vsby,wind_dir_sin,wind_dir_cos,station_observed,weather_imputed,precipitation_missing,weather_rain,weather_snow,weather_fog_mist,weather_thunder,weather_freezing,precipitation_trace,weather_qc_corrected,skyc1_CLR,skyc1_FEW,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,date,i8,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
2026-03-02 08:00:00,3,1,8,0.866025,0.5,0.0,1.0,0.866025,-0.5,2026-03-02,1,63,4.769125,19.0,2.0,19.0,1.0,1.527778,43.4125,7.25,0.0,10.0,0.869607,-0.316511,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-03-12 20:00:00,3,4,20,0.866025,0.5,0.433884,-0.900969,-0.866025,0.5,2026-03-12,0,63,4.769125,19.0,2.0,19.0,1.0,5.694444,48.06,10.5,0.0,10.0,0.339422,-0.932555,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-04-27 12:00:00,4,1,12,1.0,6.1232e-17,0.0,1.0,1.2246e-16,-1.0,2026-04-27,0,63,4.769125,19.0,2.0,19.0,1.0,16.111111,88.595,14.5,13.208,2.5,0.409337,-0.69163,1,0,0,1,0,1,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2025-12-31 20:00:00,12,3,20,-0.5,0.866025,0.974928,-0.222521,-0.866025,0.5,2025-12-31,0,63,4.769125,19.0,2.0,19.0,1.0,-5.138889,64.4475,13.5,0.0,9.0,-2.4493e-16,1.0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-01-26 08:00:00,1,1,8,0.0,1.0,0.0,1.0,0.866025,-0.5,2026-01-26,0,63,4.769125,19.0,2.0,19.0,1.0,-15.833333,51.54,11.25,0.0,10.0,-0.859447,0.496202,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2025-10-19 16:00:00,10,7,16,-1.0,-1.8370e-16,-0.781831,0.62349,-0.866025,-0.5,2025-10-19,0,63,4.769125,19.0,2.0,19.0,1.0,13.333333,44.33,10.5,0.0,10.0,-0.95143,0.210505,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2025-11-27 08:00:00,11,4,8,-0.866025,0.5,0.433884,-0.900969,0.866025,-0.5,2025-11-27,1,63,4.769125,19.0,2.0,19.0,1.0,-0.694444,49.4175,15.75,0.0,10.0,-0.892941,0.371202,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,780,780.0,780,780,2.0,2.0,2.0,2.0,10.0,10.0,10.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,5.0,5.0,15.0,15.0,15.0,15.0,"""Cash"""
2025-10-18 00:00:00,10,6,0,-1.0,-1.8370e-16,-0.974928,-0.222521,0.0,1.0,2025-10-18,0,63,4.769125,19.0,2.0,19.0,1.0,18.75,69.0075,9.5,0.0,10.0,-0.299927,-0.950971,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2025-10-18 16:00:00,10,6,16,-1.0,-1.8370e-16,-0.974928,-0.222521,-0.866025,-0.5,2025-10-18,0,63,4.769125,19.0,2.0,19.0,1.0,21.25,81.4875,6.5,0.254,10.0,0.030814,-0.73523,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,